In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import os

load_dotenv()

True

In [3]:
from src.config import load_config
from src.document_loader import load_documents
from src.chunking import create_chunks
from src.embedding import create_embedding_model, create_embeddings
from src.retrieval import create_faiss_index, retrieve
from src.llm import create_llm_client, generate_answer

c:\Users\NON_Academy_01\Desktop\GitHub\ESTP-GenAI-Group-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

config = load_config("config.yaml")

documents_directory = Path(config['documents']['directory'])
documents_pattern = config["documents"]["file_pattern"]

embedding_model_name = config['embedding']['model'] 
top_k = config['retrieval']['top_k'] 
llm_model_name =  config['llm']['model'] 
llm_temperature =  config['llm']['temperature'] 
chunk_size = config['chunking']['chunk_size'] 
chunk_overlap = config['chunking']['overlap'] 

base_url = os.getenv("LLM_BASE_URL")
api_key = os.getenv("LLM_API_KEY")


In [4]:
# ---------------------------------------------------------
# 1. Load documents
# ---------------------------------------------------------

documents = load_documents(documents_directory, documents_pattern)

print(f"Loaded {len(documents)} documents")


Loaded 3 documents


In [5]:
# ---------------------------------------------------------
# 2. Chunk documents
# ---------------------------------------------------------

chunks = create_chunks(documents, chunk_size, chunk_overlap)

print(f"Created {len(chunks)} chunks")

Created 10 chunks


In [6]:
# ---------------------------------------------------------
# 3. Create embeddings
# ---------------------------------------------------------

embedding_model =create_embedding_model(embedding_model_name)

embeddings = create_embeddings(embedding_model, chunks)

print(f"Embedding shape: {embeddings.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2509.31it/s]


Embedding shape: (10, 384)


In [7]:
# ---------------------------------------------------------
# 4. Create FAISS index
# ---------------------------------------------------------

index = create_faiss_index(embeddings)
print(f"FAISS contains {index.ntotal} vectors")


FAISS contains 10 vectors


In [8]:
# ---------------------------------------------------------
# 6. Connect to LM Studio
# ---------------------------------------------------------

client = create_llm_client(base_url, api_key)

In [9]:
question = "What is the maximum meal allowance?"

results = retrieve(
    question=question,
    embedding_model=embedding_model,
    index=index,
    chunks=chunks,
    top_k=top_k,
)

In [10]:
for result in results:
    print(f"Source: {result['source']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")
    print()
    print(result["text"])
    print("-" * 80)

Source: travel_expenses.md
Chunk ID: 0
Score: 0.7666

Travel Expenses

Employees can claim reimbursement for reasonable meals purchased during approved business trips.

The maximum meal allowance is €50 per day. The allowance covers breakfast, lunch, and dinner combined.

Taxi and public transportation costs are reimbursable when they are necessary for business travel. Employees should keep their receipts.

The company reimburses standard checked baggage fees for business trips. Additional baggage fees for personal items are not reimbursed.

Employ
--------------------------------------------------------------------------------
Source: travel_policy.md
Chunk ID: 0
Score: 0.6289

Business Travel Policy

Employees may travel for business when the trip is approved by their manager.

Flights longer than six hours may be booked in business class. Flights of six hours or less should be booked in economy class.

Employees should normally book hotels with a nightly cost of no more than €180. E

In [11]:
answer = generate_answer(
    question=question,
    results=results,
    client=client,
    model=llm_model_name,
    temperature=llm_temperature,
)

print(answer)

The maximum meal allowance is €50 per day.
